# Pothole & Road Surface Damage — Kaggle Pipeline

Mirrors `scripts/` end to end on Kaggle's GPU runtime, where the RDD2022 data and CUDA actually live.
Local (Apple Silicon / MPS) is for iterating on script logic only — real training runs happen here.

Importing just this notebook via Kaggle's GitHub integration does NOT bring along `scripts/` —
Kaggle only pulls the single `.ipynb`. The next cell clones the repo instead. Attach the RDD2022
dataset as an Input separately.

In [ ]:
import os

REPO_DIR = "/kaggle/working/pothole_cv"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/kabirarshafidz/pothole_cv.git {REPO_DIR}

%pip install -q -r {REPO_DIR}/requirements-kaggle.txt

In [ ]:
import sys

sys.path.append(f"{REPO_DIR}/scripts")

import torch

from utils import CLASS_NAMES, COUNTRIES, DATA_ROOT, OUTPUTS_ROOT, RUNS_ROOT, get_device

print("device:", get_device())
print("cuda available:", torch.cuda.is_available())
print("DATA_ROOT:", DATA_ROOT)

## 1. Data prep

RDD_SPLIT is already YOLO-formatted (`.txt` labels) and already split 70/15/15, with all 6
countries mixed together in each of train/val/test. Filenames carry a country prefix
(`Japan_000123.jpg`) — `build_country_lists` filters by that prefix into per-country image
list files under `outputs/splits/`, which is how we get a Japan-only training set and
per-country test sets for the generalization study. No XML conversion or re-splitting needed
here (and re-splitting would just reshuffle a split someone else already froze).

In [ ]:
from build_country_lists import build_lists

build_lists()

# Sanity-check the pre-converted label class order matches CLASS_NAMES (D00, D10, D20, D40).
# If class ids in these files don't look like 0-3, or D40 (pothole) AP comes back near zero
# later, the upstream conversion used a different order than we assume.
sample_label = next((DATA_ROOT / "train" / "labels").glob("Japan_*.txt"))
print(sample_label.name)
print(sample_label.read_text())

In [ ]:
DRY_RUN = False        # True -> tiny subset, 1 epoch, to smoke-test the pipeline fast
FORCE_RETRAIN = False  # True -> retrain even if best.pt already exists on disk

from train import make_dataset_yaml

dataset_yaml_path = make_dataset_yaml("Japan", limit=40 if DRY_RUN else None)
print(dataset_yaml_path)

## 2. Train (YOLOv8s, Japan baseline)

Expected mAP@50: 0.60–0.67. Above 0.80 almost certainly means train/test overlap — recheck the split.

If `runs/japan_baseline/weights/best.pt` already exists, this cell reuses it instead of
retraining — a bug in a later cell shouldn't cost another multi-hour run. Set
`FORCE_RETRAIN = True` above to retrain anyway, or `DRY_RUN = True` to validate the whole
notebook end to end on a tiny subset (1 epoch) before committing to a real run.

In [ ]:
from ultralytics import YOLO

best_weights = RUNS_ROOT / "japan_baseline" / "weights" / "best.pt"

if best_weights.exists() and not FORCE_RETRAIN:
    print(f"Reusing existing weights at {best_weights} (set FORCE_RETRAIN=True to retrain)")
else:
    model = YOLO("yolov8s.pt")
    model.train(
        data=dataset_yaml_path,
        imgsz=640,
        batch=16,
        epochs=1 if DRY_RUN else 50,
        patience=10,
        device=get_device(),
        project=str(RUNS_ROOT),
        name="japan_baseline",
    )

## 3. Cross-country generalization

Evaluate the Japan-trained model on all 6 countries. Expect a 15–25 mAP point drop
Japan → India, with D20 (alligator crack) transferring worst — that's the core finding, not a bug.

In [ ]:
import csv

from evaluate import make_test_yaml

eval_model = YOLO(str(best_weights))
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
results_path = OUTPUTS_ROOT / "generalization_results.csv"

with open(results_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["country", "map50", "map50_95", "precision", "recall"])
    for country in COUNTRIES:
        data_yaml = make_test_yaml(country, limit=20 if DRY_RUN else None)
        metrics = eval_model.val(data=data_yaml, device=get_device())
        writer.writerow([country, metrics.box.map50, metrics.box.map, metrics.box.mp, metrics.box.mr])

print(f"Wrote {results_path}")

## 4. Severity scoring (area vs depth)

Compares the area-based baseline against Depth Anything-assisted severity on a sample test image.

In [ ]:
from severity import area_severity, depth_severity
from PIL import Image
from transformers import pipeline
import numpy as np

sample_image_path = next((DATA_ROOT / "test" / "images").glob("Japan_*.jpg"))
sample_image = Image.open(sample_image_path).convert("RGB")
detections = eval_model.predict(sample_image, device=get_device())[0]

depth_pipe = pipeline("depth-estimation", model="LiheYoung/depth-anything-small-hf", device=get_device())
depth_map = np.array(depth_pipe(sample_image)["depth"])

for box in detections.boxes:
    xyxy = box.xyxy[0].tolist()
    a = area_severity(xyxy, sample_image.size[::-1])
    d = depth_severity(xyxy, depth_map)
    print(f"class={int(box.cls[0])} area_severity={a:.4f} depth_severity={d:.4f}")

## 5. Demo visualization

Annotated image with severity-colored boxes (green=low, orange=medium, red=high).

In [ ]:
import cv2
import matplotlib.pyplot as plt

from visualize import SEVERITY_COLORS, bucket

image_bgr = cv2.imread(str(sample_image_path))
for box in detections.boxes:
    x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
    score = area_severity((x1, y1, x2, y2), image_bgr.shape[:2])
    color = SEVERITY_COLORS[bucket(score)]
    cv2.rectangle(image_bgr, (x1, y1), (x2, y2), color, 2)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()